# Anotaciones en revisión: qué clase ve el detector

`prepare_annotations.py` marca `requires_review` toda fila que no se puede asignar con certeza a
un par del vocabulario (`CALL_TYPES` en `src/data/species.py`). No entran al experimento, ni como
caja ni como fondo. Hay tres motivos:

| motivo | qué pasó en la tabla de Raven |
|---|---|
| fuera del vocabulario | un tipo de llamada que no es de esa especie |
| sin tipo de llamada | `Call Type` vacío |
| especie ≠ carpeta | `Species` escrita distinta de la carpeta donde está la grabación |

Para cada una se le pregunta a un modelo que **no vio esa grabación** qué hay en su caja: la
predicción de mayor score que la solapa (IoU ≥ 0,3) en una ventana de 3 s centrada en ella. Si la
grabación está en train responde el pliegue del k-fold que la dejó fuera (`runs/yolo26s_v3_kfold5`,
siempre que el k-fold sea del caché actual); si no, el modelo final.

Salen dos tablas: un resumen por motivo y clase escrita, y la lista fila por fila para volver a la
tabla. `clase escrita` es la especie escrita (la de la carpeta si `Species` venía vacía) más el
tipo de llamada normalizado.

Lee `data/cleaned/`, el caché (`data/processed/`) y `runs/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from analysis.predictions import (
    centered_window,
    class_scores,
    detect_clip,
    fold_checkpoints,
    load_model,
    window_box,
)
from core.config import CLEANED_DIR, RAW_DIR, RUNS_DIR
from data.annotations import SPECIES_CODES, species_of

RUN, KFOLD = "yolo26s_v3", "yolo26s_v3_kfold5"
MATCH_IOU = 0.3
DEVICE = "cpu"  # ~1 300 ventanas: no hace falta GPU
pd.set_option("display.width", 160)

## Las filas en revisión

Cada fila conserva de qué tabla de `cleaned/` viene y su número de fila, para poder volver a ella.

In [2]:
def read_tables() -> pd.DataFrame:
    # Las tablas de `cleaned/` de las especies del experimento, con su ruta y el número de fila
    frames = []
    for path in sorted(CLEANED_DIR.rglob("*.txt")):
        table = pd.read_csv(path, sep="\t", keep_default_na=False)
        if species_of(path) not in SPECIES_CODES or table.empty:
            continue
        relative = path.relative_to(CLEANED_DIR)
        table["tabla"], table["fila"] = str(relative), np.arange(len(table))
        table["audio"] = str((RAW_DIR / relative).with_suffix(".wav"))
        frames.append(table)
    return pd.concat(frames, ignore_index=True)


annotations = read_tables()
review = annotations[annotations["requires_review"]].copy()
other_species = review["written_species"].ne("") & review["written_species"].ne(review["species"])
review["motivo"] = np.select(
    [other_species, review["call_type"].eq("")],
    ["especie ≠ carpeta", "sin tipo de llamada"],
    "fuera del vocabulario",
)
review["clase escrita"] = (
    np.where(other_species, review["written_species"], review["species"])
    + "/"
    + review["call_type"]
)
print(f"{len(review)} de {len(annotations)} anotaciones en revisión")
review["motivo"].value_counts()

1276 de 19213 anotaciones en revisión


motivo
fuera del vocabulario    969
sin tipo de llamada      179
especie ≠ carpeta        128
Name: count, dtype: int64

## Qué ve el modelo en cada una

Por fila: el checkpoint que no vio su grabación, una ventana de 3 s centrada en la anotación, y de
las detecciones que solapan la caja, la de mayor score (`nada` si ninguna la solapa).

In [3]:
checkpoint_of = fold_checkpoints(KFOLD)  # grabación de train -> pliegue que la dejó fuera
FINAL = RUNS_DIR / RUN / "best.pt"
print(f"{len(checkpoint_of)} grabaciones de train con pliegue; el resto usa {FINAL.parent.name}")


def seen_by_model(row: pd.Series) -> dict[str, object]:
    checkpoint = checkpoint_of.get(Path(row["audio"]).stem, FINAL)
    loaded = load_model(checkpoint, DEVICE)
    begin, end = float(row["begin_time_s"]), float(row["end_time_s"])
    start = centered_window(begin, end)
    target = window_box(begin, end, float(row["low_freq_hz"]), float(row["high_freq_hz"]), start)
    detections = detect_clip(loaded, row["audio"], start, DEVICE)
    scores = class_scores(detections, target, len(loaded.labels), MATCH_IOU)[0]
    best = int(scores.argmax())
    return {
        "tabla": row["tabla"],
        "fila": row["fila"],
        "motivo": row["motivo"],
        "modelo": checkpoint.parent.name,
        "clase escrita": row["clase escrita"],
        "clase detectada": loaded.labels.name(best) if scores[best] > 0 else "nada",
        "score": round(float(scores[best]), 2),
    }


verdict = pd.DataFrame([seen_by_model(row) for _, row in review.iterrows()])

1248 grabaciones de train con pliegue; el resto usa yolo26s_v3


Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


Overriding model.yaml nc=80 with nc=25


## Resumen por motivo y clase escrita

`ve`: las tres clases detectadas más frecuentes para esa clase escrita, con cuántas filas cada una.
Los motivos van de más a menos filas y, dentro de cada uno, las clases escritas también.

In [4]:
by_motivo = verdict["motivo"].value_counts()
verdict["motivo"] = verdict["motivo"].astype(pd.CategoricalDtype(by_motivo.index))


def top_seen(detected: pd.Series) -> str:
    return ", ".join(f"{c} {n}" for c, n in detected.value_counts().head(3).items())


(
    verdict.groupby(["motivo", "clase escrita"], observed=True)
    .agg(filas=("fila", "size"), ve=("clase detectada", top_seen))
    .sort_values(["motivo", "filas"], ascending=[True, False])
)

filas                             ve
motivo                clase escrita                                                     
fuera del vocabulario pt/sqc                          387    pt/sqc 381, nada 5, pt/dc 1
                      lw/tj                           164  lw/trino 141, nada 9, lw/ta 7
                      as/cp                            91     as/hc 70, nada 20, as/bc 1
                      as/pp                            85              as/hc 57, nada 28
                      as/ip                            73     as/hc 44, nada 27, as/bc 1
                      pt/pp                            43               pt/dc 39, nada 4
                      pt/bp                            39               pt/dc 35, nada 4
                      lw/a                             20  nada 17, sb/ppc 2, lw/trino 1
                      lw/b                             15      sm/pc 7, nada 5, sb/ppc 3
                      sb/pcs                            8               nada 7, sb/ppc 1
                      sm/hc                             8     nada 5, sm/hic 2, sb/spc 1
                      aa/sqc                            6                        aa/sc 6
                      as/cc                             4                as/hc 3, nada 1
                      sm/sic                            4               sm/sc 3, sm/pc 1
                      as/pc                             3                        as/hc 3
                      lw/c                              3                     lw/trino 3
                      sb/phc                            3                       sb/ppc 3
                      sm/chc                            3                         nada 3
                      aa/hf                             1                        aa/gc 1
                      as/chc                            1                        as/hc 1
                      lw/chc                            1                       sb/ppc 1
                      lw/lw                             1                        lw/vc 1
                      lw/t                              1                        lw/ta 1
                      pt/chc                            1                         nada 1
                      pt/d                              1                         nada 1
                      sb/cc                             1                       sb/pcc 1
                      sb/hic                            1                       sb/spc 1
                      sm/spc                            1                       sm/hic 1
sin tipo de llamada   as/                              53     nada 27, as/hc 22, as/bc 4
                      sm/                              47      sm/cc 31, nada 5, sm/fs 5
                      pt/                              29     pt/dc 20, nada 7, pt/sqc 1
                      ac/                              21    ac/bc 10, ac/chc 6, ac/gc 3
                      lw/                              15       lw/cs 6, nada 5, lw/ta 1
                      sb/                              11      sb/ppc 5, nada 3, sb/sc 1
                      cc/                               2                        sm/cc 2
                      aa/                               1                        aa/gc 1
especie ≠ carpeta     as/bc                            34               ac/bc 32, nada 2
                      ac/hc                            23               nada 18, as/hc 5
                      ac/bc                            14                       as/bc 14
                      sm/ppc                           11     sb/ppc 9, nada 1, sb/pcc 1
                      sn/cc                             7                        sm/cc 7
                      as/chc                            6                ac/bc 5, nada 1
                      sm/cc                             6                sm/cc 5, nada 1
                      ac/sc                    

## Fila por fila

Para ir a la tabla de Raven: `tabla` es la ruta dentro de `data/cleaned/` y `fila` el índice
(desde 0) dentro de ella.

In [5]:
pd.set_option("display.max_rows", len(verdict))
verdict.sort_values(["motivo", "tabla", "fila"]).reset_index(drop=True)

,tabla,fila,motivo,modelo,clase escrita,clase detectada,score
0,bolivian_squirrel_monkey__SB/20240202_165553.txt,0,fuera del vocabulario,fold4,sb/cc,sb/pcc,0.04
1,bolivian_squirrel_monkey__SB/20240430_165406.txt,2,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
2,bolivian_squirrel_monkey__SB/20240430_165406.txt,3,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
3,bolivian_squirrel_monkey__SB/20240430_165406.txt,4,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
4,bolivian_squirrel_monkey__SB/20240430_165406.txt,5,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
5,bolivian_squirrel_monkey__SB/20240430_165443.txt,1,fuera del vocabulario,yolo26s_v3,sb/pcs,sb/ppc,0.04
6,bolivian_squirrel_monkey__SB/20240430_165443.txt,2,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
7,bolivian_squirrel_monkey__SB/20240430_165443.txt,3,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
8,bolivian_squirrel_monkey__SB/20240430_165443.txt,4,fuera del vocabulario,yolo26s_v3,sb/pcs,nada,0.00
9,bolivian_squirrel_monkey__SB/20240531_115533.txt,4,fuera del vocabulario,fold4,sb/hic,sb/spc,0.46
